In [3]:
import pymupdf
import pymupdf4llm
from tqdm import tqdm
import os

PDF_PATH = "data/raw/MachineLearningTomMitchell.pdf"
OUTPUT_PATH = "data/processed/ml_book.md"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

print("Opening PDF...")

doc = pymupdf.open(PDF_PATH)

print(f"Total pages: {len(doc)}")
print("Starting extraction...\n")


all_pages = []

for page_num in tqdm(
    range(len(doc)),
    desc="Extracting PDF",
    unit="page"
):
    # Extract one page using PyMuPDF4LLM
    page_data = pymupdf4llm.to_markdown(
        doc,
        pages=[page_num],
        page_chunks=True
    )

    all_pages.extend(page_data)


doc.close()


print("\nSaving Markdown...")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:

    for page_number, page in enumerate(all_pages, start=1):

        f.write(
            f"\n\n<!-- PAGE {page_number} -->\n\n"
        )

        f.write(page["text"])


print("\nExtraction completed!")
print(f"Saved to: {OUTPUT_PATH}")

Opening PDF...
Total pages: 421
Starting extraction...



Extracting PDF: 100%|██████████| 421/421 [06:27<00:00,  1.09page/s]


Saving Markdown...

Extraction completed!
Saved to: data/processed/ml_book.md


In [4]:
from pathlib import Path

OUTPUT_PATH = Path("data/processed/ml_book.md")

text = OUTPUT_PATH.read_text(encoding="utf-8")

print(f"Characters: {len(text):,}")
print(f"Words: {len(text.split()):,}")
print(f"Lines: {len(text.splitlines()):,}")
print(text[:5000])

Characters: 1,134,785
Words: 174,545
Lines: 9,602


<!-- PAGE 1 -->





<!-- PAGE 2 -->

# Machine Learning 

## Tom M. Mitchell 

### **Product Details** 

- **Hardcover:** 432 pages ; Dimensions (in inches): 0.75 x 10.00 x 6.50 

- **Publisher:** McGraw-Hill Science/Engineering/Math; (March 1, 1997) 

- **ISBN:** 0070428077 

- **Average Customer Review:** Based on 16 reviews. 

- **Amazon.com Sales Rank:** 42,816 

- **Popular in:** <u>Redmond, WA (#17)</u> , <u>Ithaca, NY (#9)</u> 

#### **Editorial Reviews** 

**_From Book News, Inc._** An introductory text on primary approaches to machine learning and the study of computer algorithms that improve automatically through experience. Introduce basics concepts from statistics, artificial intelligence, information theory, and other disciplines as need arises, with balanced coverage of theory and practice, and presents major algorithms with illustrations of their use. Includes chapter exercises. Online data sets and implementations of 

In [5]:
middle = len(text) // 2

print(text[middle:middle + 5000])

f course this is difficult to assure if one does not know **_C_** in advance (what is **_C_** for a program that must learn to recognize faces from images?), unless H is taken to be the power set of X. As pointed out in Chapter 2, such an unbiased **_H_** will not support accurate generalization from a reasonable number of training examples. / Nevertheless, the results based on the PAC learning model provide useful insights regarding the relative complexity of different learning problems and regarding the rate at which generalization accuracy improves with additional training examples. Furthermore, in Section **7.3.1** we will lift this restrictive assumption, to consider the case in which the learner makes no prior assumption about the form of the target concept. 

# **7.3 SAMPLE COMPLEXITY FOR FINITE HYPOTHESIS SPACES** 

As noted above, PAC-learnability is largely determined by the number of training examples required by the learner. The growth in the number of required training exa

In [ ]:
import re

pages = re.findall(r"<!-- PAGE (\d+) -->", text)

print(f"Page markers found: {len(pages)}")
print("First 10:", pages[:10])
print("Last 10:", pages[-10:])

Page markers found: 421
First 10: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
Last 10: ['412', '413', '414', '415', '416', '417', '418', '419', '420', '421']


In [8]:
from pathlib import Path
import re

INPUT_PATH = Path("data/processed/ml_book.md")
OUTPUT_PATH = Path("data/processed/clean_ml_book.md")


# --------------------------------------------------
# Load
# --------------------------------------------------

text = INPUT_PATH.read_text(encoding="utf-8")

print(f"Original characters: {len(text):,}")


# --------------------------------------------------
# 1. Normalize line endings
# --------------------------------------------------

text = text.replace("\r\n", "\n")
text = text.replace("\r", "\n")


# --------------------------------------------------
# 2. Remove excessive spaces
# --------------------------------------------------

text = re.sub(r"[ \t]+", " ", text)


# --------------------------------------------------
# 3. Remove excessive blank lines
# --------------------------------------------------

text = re.sub(r"\n{3,}", "\n\n", text)


# --------------------------------------------------
# 4. Clean spaces around our page markers
# --------------------------------------------------

text = re.sub(
    r"\n*\s*(<!-- PAGE \d+ -->)\s*\n*",
    r"\n\n\1\n\n",
    text
)


# --------------------------------------------------
# 5. Remove blank spaces before punctuation
# --------------------------------------------------

text = re.sub(
    r" +([,.!?;:])",
    r"\1",
    text
)


# --------------------------------------------------
# Save
# --------------------------------------------------

OUTPUT_PATH.write_text(
    text.strip(),
    encoding="utf-8"
)

print(f"Cleaned characters: {len(text):,}")
print(f"Saved to: {OUTPUT_PATH}")


Original characters: 1,134,785
Cleaned characters: 1,131,408
Saved to: data\processed\clean_ml_book.md


In [10]:
from pathlib import Path

OUTPUT_PATH = Path("data/processed/clean_ml_book.md")

text = OUTPUT_PATH.read_text(encoding="utf-8")

print(f"Characters: {len(text):,}")
print(f"Words: {len(text.split()):,}")
print(f"Lines: {len(text.splitlines()):,}")
print(text[:5000])

Characters: 1,131,404
Words: 173,680
Lines: 8,539
<!-- PAGE 1 -->



<!-- PAGE 2 -->

# Machine Learning 

## Tom M. Mitchell 

### **Product Details** 

- **Hardcover:** 432 pages; Dimensions (in inches): 0.75 x 10.00 x 6.50 

- **Publisher:** McGraw-Hill Science/Engineering/Math; (March 1, 1997) 

- **ISBN:** 0070428077 

- **Average Customer Review:** Based on 16 reviews. 

- **Amazon.com Sales Rank:** 42,816 

- **Popular in:** <u>Redmond, WA (#17)</u>, <u>Ithaca, NY (#9)</u> 

#### **Editorial Reviews** 

**_From Book News, Inc._** An introductory text on primary approaches to machine learning and the study of computer algorithms that improve automatically through experience. Introduce basics concepts from statistics, artificial intelligence, information theory, and other disciplines as need arises, with balanced coverage of theory and practice, and presents major algorithms with illustrations of their use. Includes chapter exercises. Online data sets and implementations of severa

In [2]:
headings = re.findall(
    r"(?m)^(#{1,6})\s+(.+?)\s*$",
    text
)

print(f"Total headings: {len(headings)}\n")

for level, title in headings[:100]:
    print(f"{len(level)} | {title}")

Total headings: 365

1 | Machine Learning
2 | Tom M. Mitchell
3 | **Product Details**
4 | **Editorial Reviews**
1 | **PREFACE**
1 | **ACKNOWLEDGMENTS**
1 | **1.1 WELL-POSED LEARNING PROBLEMS**
1 | **A checkers learning problem:**
2 | Control theory
2 | Philosophy
1 | **A robot driving learning problem:**
1 | **1.2 DESIGNING A LEARNING SYSTEM**
1 | **1.2.1 Choosing the Training Experience**
1 | **A checkers learning problem:**
1 | **1.2.2 Choosing the Target Function**
1 | **1.23 Choosing a Representation for the Target Function**
1 | **1.2.4 Choosing a Function Approximation Algorithm**
1 | **1.2.4.1 ESTIMATING TRAINING VALUES**
1 | **~ u l k for estimating training values.**
1 | **1.2.4.2 ADJUSTING THE WEIGHTS**
2 | **LMS weight update rule.**
1 | **1.2.5 The Final Design**
1 | **1.3 PERSPECTIVES AND ISSUES IN MACHINE LEARNING**
1 | **1.3.1 Issues in Machine Learning**
1 | **1.4 HOW TO READ THIS BOOK**
1 | **1.5 SUMMARY AND FURTHER READING**
1 | **EXERCISES**
1 | **REFERENCES**
1 | **

In [3]:
lines = text.splitlines()

current_page = None
current_heading = None

structure = []

for line in lines:

    # Page marker
    page_match = re.match(
        r"<!-- PAGE (\d+) -->",
        line.strip()
    )

    if page_match:
        current_page = int(page_match.group(1))

    # Heading
    heading_match = re.match(
        r"^(#{1,6})\s+(.+?)\s*$",
        line.strip()
    )

    if heading_match:

        level = len(heading_match.group(1))
        title = heading_match.group(2)

        structure.append({
            "page": current_page,
            "level": level,
            "title": title
        })

In [4]:
for item in structure[:50]:
    print(item)

{'page': 2, 'level': 1, 'title': 'Machine Learning'}
{'page': 2, 'level': 2, 'title': 'Tom M. Mitchell'}
{'page': 2, 'level': 3, 'title': '**Product Details**'}
{'page': 2, 'level': 4, 'title': '**Editorial Reviews**'}
{'page': 3, 'level': 1, 'title': '**PREFACE**'}
{'page': 4, 'level': 1, 'title': '**ACKNOWLEDGMENTS**'}
{'page': 14, 'level': 1, 'title': '**1.1 WELL-POSED LEARNING PROBLEMS**'}
{'page': 15, 'level': 1, 'title': '**A checkers learning problem:**'}
{'page': 16, 'level': 2, 'title': 'Control theory'}
{'page': 16, 'level': 2, 'title': 'Philosophy'}
{'page': 16, 'level': 1, 'title': '**A robot driving learning problem:**'}
{'page': 17, 'level': 1, 'title': '**1.2 DESIGNING A LEARNING SYSTEM**'}
{'page': 17, 'level': 1, 'title': '**1.2.1 Choosing the Training Experience**'}
{'page': 18, 'level': 1, 'title': '**A checkers learning problem:**'}
{'page': 19, 'level': 1, 'title': '**1.2.2 Choosing the Target Function**'}
{'page': 20, 'level': 1, 'title': '**1.23 Choosing a Repres

In [1]:
from pathlib import Path
import re
import tiktoken


INPUT_PATH = Path("data/processed/clean_ml_book.md")
OUTPUT_PATH = Path("data/processed/manual_chunks.json")


CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

SOURCE_NAME = "MachineLearningTomMitchell.pdf"


# --------------------------------------------------
# Load document
# --------------------------------------------------

text = INPUT_PATH.read_text(encoding="utf-8")

print(f"Loaded characters: {len(text):,}")


# --------------------------------------------------
# Tokenizer
# --------------------------------------------------

tokenizer = tiktoken.get_encoding("cl100k_base")


def count_tokens(text):
    return len(tokenizer.encode(text))


def split_into_token_chunks(text, chunk_size, overlap):

    tokens = tokenizer.encode(text)

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + chunk_size

        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(chunk_tokens)

        chunks.append(chunk_text)

        start += chunk_size - overlap

    return chunks

Loaded characters: 1,131,404


In [2]:
lines = text.splitlines()

current_page = None
current_chapter = None
current_section = None

sections = []

current_content = []


def save_section():

    if not current_content:
        return

    content = "\n".join(current_content).strip()

    if content:
        sections.append({
            "page": current_page,
            "chapter": current_chapter,
            "section": current_section,
            "text": content
        })


for line in lines:

    stripped = line.strip()

    # -----------------------------
    # Page
    # -----------------------------

    page_match = re.match(
        r"<!-- PAGE (\d+) -->",
        stripped
    )

    if page_match:

        save_section()

        current_content = []

        current_page = int(page_match.group(1))

        continue


    # -----------------------------
    # Heading
    # -----------------------------

    heading_match = re.match(
        r"^(#{1,6})\s+(.+?)\s*$",
        stripped
    )

    if heading_match:

        level = len(heading_match.group(1))
        title = heading_match.group(2).strip()


        # H1 / H2 → chapter-level context
        if level <= 2:

            if current_content:
                save_section()

                current_content = []

            current_chapter = title
            current_section = None

        else:

            if current_content:
                save_section()

                current_content = []

            current_section = title

        continue


    # -----------------------------
    # Normal content
    # -----------------------------

    current_content.append(line)


# Save final section
save_section()


print(f"Sections created: {len(sections):,}")

Sections created: 730


In [3]:
chunks = []

chunk_counter = 0


for section in sections:

    section_text = section["text"]

    section_chunks = split_into_token_chunks(
        section_text,
        CHUNK_SIZE,
        CHUNK_OVERLAP
    )


    for chunk_index, chunk_text in enumerate(section_chunks):

        chunk_counter += 1

        chunks.append({

            "chunk_id": f"ml_{chunk_counter:06d}",

            "text": chunk_text,

            "metadata": {

                "source": SOURCE_NAME,

                "page": section["page"],

                "chapter": section["chapter"],

                "section": section["section"],

                "chunk_index": chunk_index,

                "token_count": count_tokens(chunk_text)
            }
        })


print(f"Total chunks: {len(chunks):,}")

Total chunks: 747


In [4]:
import json


with open(OUTPUT_PATH, "w", encoding="utf-8") as f:

    json.dump(
        chunks,
        f,
        indent=2,
        ensure_ascii=False
    )


print(f"Saved chunks to: {OUTPUT_PATH}")

Saved chunks to: data\processed\manual_chunks.json


In [5]:
print(json.dumps(
    chunks[0],
    indent=2,
    ensure_ascii=False
))

{
  "chunk_id": "ml_000001",
  "text": "- **Hardcover:** 432 pages; Dimensions (in inches): 0.75 x 10.00 x 6.50 \n\n- **Publisher:** McGraw-Hill Science/Engineering/Math; (March 1, 1997) \n\n- **ISBN:** 0070428077 \n\n- **Average Customer Review:** Based on 16 reviews. \n\n- **Amazon.com Sales Rank:** 42,816 \n\n- **Popular in:** <u>Redmond, WA (#17)</u>, <u>Ithaca, NY (#9)</u>",
  "metadata": {
    "source": "MachineLearningTomMitchell.pdf",
    "page": 2,
    "chapter": "Tom M. Mitchell",
    "section": "**Product Details**",
    "chunk_index": 0,
    "token_count": 116
  }
}


In [6]:
for chunk in chunks[:3]:

    print("=" * 80)

    print("ID:", chunk["chunk_id"])
    print("Page:", chunk["metadata"]["page"])
    print("Chapter:", chunk["metadata"]["chapter"])
    print("Section:", chunk["metadata"]["section"])
    print("Tokens:", chunk["metadata"]["token_count"])

    print("\nTEXT:\n")
    print(chunk["text"][:1000])

ID: ml_000001
Page: 2
Chapter: Tom M. Mitchell
Section: **Product Details**
Tokens: 116

TEXT:

- **Hardcover:** 432 pages; Dimensions (in inches): 0.75 x 10.00 x 6.50 

- **Publisher:** McGraw-Hill Science/Engineering/Math; (March 1, 1997) 

- **ISBN:** 0070428077 

- **Average Customer Review:** Based on 16 reviews. 

- **Amazon.com Sales Rank:** 42,816 

- **Popular in:** <u>Redmond, WA (#17)</u>, <u>Ithaca, NY (#9)</u>
ID: ml_000002
Page: 2
Chapter: Tom M. Mitchell
Section: **Editorial Reviews**
Tokens: 242

TEXT:

**_From Book News, Inc._** An introductory text on primary approaches to machine learning and the study of computer algorithms that improve automatically through experience. Introduce basics concepts from statistics, artificial intelligence, information theory, and other disciplines as need arises, with balanced coverage of theory and practice, and presents major algorithms with illustrations of their use. Includes chapter exercises. Online data sets and implementations 

In [1]:
from pathlib import Path
import re

INPUT_PATH = Path("data/processed/clean_ml_book.md")

text = INPUT_PATH.read_text(encoding="utf-8")
lines = text.splitlines()


# ============================================================
# 1. Patterns
# ============================================================

# Example:
# CHAPTER 1
# CHAPTER 1 INTRODUCTION
# CHAPTER 6 BAYESIAN LEARNING

chapter_pattern = re.compile(
    r"^\s*CHAPTER\s+(\d{1,2})(?:\s+(.*?))?\s*$",
    re.IGNORECASE
)


# Example:
# 1.1 What Is Machine Learning?
# 6.9.1 An Illustrative Example
# 6.12 THE EM ALGORITHM

section_pattern = re.compile(
    r"^\s*(\d{1,2}(?:\.\d+)+)\s+(.+?)\s*$"
)


# ============================================================
# 2. Helper functions
# ============================================================

def clean_title(title):
    """
    Remove Markdown artifacts and trailing page numbers.
    """

    # Remove Markdown bold
    title = re.sub(r"\*\*", "", title)

    # Remove HTML tags
    title = re.sub(r"<[^>]+>", "", title)

    # Remove trailing page number
    title = re.sub(r"\s+\d{1,4}\s*$", "", title)

    return title.strip()


def looks_like_toc_entry(title):
    """
    Detect obvious Table-of-Contents entries.
    """

    # TOC entries often contain a trailing page number
    if re.search(r"\*\*\s*\d{1,4}\s*\*\*$", title):
        return True

    # Also catch plain trailing page numbers
    if re.search(r"\s+\d{1,4}\s*$", title):
        return True

    return False


# ============================================================
# 3. State
# ============================================================

current_page = None

current_chapter_number = None
current_chapter = None

current_section_number = None
current_section = None

current_content = []

sections = []

book_started = False


# ============================================================
# 4. Save current section
# ============================================================

def save_current_section():

    global current_content

    content = "\n".join(current_content).strip()

    if not content:
        current_content = []
        return

    if not book_started:
        current_content = []
        return

    sections.append({
        "page": current_page,

        "chapter_number": current_chapter_number,
        "chapter": current_chapter,

        "section_number": current_section_number,
        "section": current_section,

        "text": content
    })

    current_content = []


# ============================================================
# 5. Parse
# ============================================================

for i, line in enumerate(lines):

    stripped = line.strip()


    # --------------------------------------------------------
    # PAGE
    # --------------------------------------------------------

    page_match = re.match(
        r"<!-- PAGE (\d+) -->",
        stripped
    )

    if page_match:

        current_page = int(page_match.group(1))

        continue


    # --------------------------------------------------------
    # CHAPTER
    # --------------------------------------------------------

    chapter_match = chapter_pattern.match(stripped)

    if chapter_match:

        chapter_number = int(
            chapter_match.group(1)
        )

        raw_title = (
            chapter_match.group(2) or ""
        ).strip()


        # --------------------------------------------
        # If title exists on same line
        # --------------------------------------------

        if raw_title:

            # Skip obvious TOC entry
            if looks_like_toc_entry(raw_title):

                continue

            chapter_title = clean_title(
                raw_title
            )


        # --------------------------------------------
        # Otherwise look at next non-empty line
        # --------------------------------------------

        else:

            chapter_title = None

            for next_line in lines[i + 1:i + 5]:

                candidate = next_line.strip()

                if not candidate:
                    continue

                # Skip page-number-only lines
                if re.fullmatch(
                    r"\**\s*\d{1,4}\s*\**",
                    candidate
                ):
                    continue

                chapter_title = clean_title(candidate)

                break


            if not chapter_title:
                continue


        # --------------------------------------------
        # Reject TOC-like entries
        # --------------------------------------------

        if looks_like_toc_entry(raw_title):
            continue


        # --------------------------------------------
        # New actual chapter
        # --------------------------------------------

        save_current_section()

        current_chapter_number = chapter_number

        current_chapter = chapter_title

        current_section_number = None

        current_section = None

        book_started = True

        continue


    # --------------------------------------------------------
    # SECTION
    # --------------------------------------------------------

    section_match = section_pattern.match(
        stripped
    )

    if (
        section_match
        and book_started
        and current_chapter_number is not None
    ):

        section_number = (
            section_match.group(1)
        )

        section_title = clean_title(
            section_match.group(2)
        )


        # Make sure section belongs
        # to current chapter

        section_chapter = int(
            section_number.split(".")[0]
        )

        if (
            section_chapter
            == current_chapter_number
        ):

            save_current_section()

            current_section_number = (
                section_number
            )

            current_section = (
                section_title
            )

            continue


    # --------------------------------------------------------
    # NORMAL CONTENT
    # --------------------------------------------------------

    if book_started:

        current_content.append(line)


# Save final section
save_current_section()


print(
    f"Sections extracted: {len(sections):,}"
)

Sections extracted: 2


In [2]:
for section in sections[:20]:

    print("=" * 80)

    print("Page:", section["page"])
    print(
        "Chapter:",
        section["chapter_number"],
        section["chapter"]
    )

    print(
        "Section:",
        section["section_number"],
        section["section"]
    )

    print()
    print(section["text"][:300])

Page: 109
Chapter: 3 DECISION TREE LEARNING _59_
Section: None None

**Which attribute is the best classifier?** 

<!-- Start of picture text -->
S: [9+,5-I S: [9+,5-I<br>E =0.940<br>Humidity<br>High wx E S.940 Strong<br>[3+,4-I [6t,l-l [6+,2-I [3+,3-I<br>E S.985 E S.592 ES.811 E =1.00<br>Gain (S, Hurnidiry ) Gain (S, Wind)<br>=,940 - (8/14).811 - (6114)l.O<br>=,048<
Page: 421
Chapter: 4 ARTIFICIAL NEURAL NETWORKS _97_
Section: None None

continuous function of its input. More precisely, the sigmoid unit computes its output **_o_** as 

where 

a is often called the sigmoid function or, alternatively, the logistic function. Note its output ranges between 0 and 1, increasing monotonically with its input (see the threshold function plo


In [3]:
chapters = {}

for section in sections:

    key = (
        section["chapter_number"],
        section["chapter"]
    )

    chapters[key] = True


print("Chapters detected:\n")

for number, title in chapters:

    print(f"Chapter {number}: {title}")

Chapters detected:

Chapter 3: DECISION TREE LEARNING _59_
Chapter 4: ARTIFICIAL NEURAL NETWORKS _97_


In [4]:
for section in sections[:5]:

    print(
        section["chapter"],
        "|",
        section["section"]
    )

DECISION TREE LEARNING _59_ | None
ARTIFICIAL NEURAL NETWORKS _97_ | None
